# 2 · QA-level multimodal embeddings (Qwen2.5-Omni)

Encodes every QA unit with **Qwen2.5-Omni-7B** (4-bit NF4, bfloat16 compute, double quantization): the prompt carries the topic, question, answer and supporting transcript lines, together with the audio clips of those lines. The embedding is the masked mean of the Thinker's last hidden states (3584 dimensions; paper, Section IV-A3).

| Section | Variant (paper) | Output file suffix |
|---|---|---|
| A | QA Text + Audio, **General Prompt** (ablation) | `_embeddings.pkl` |
| B | **QA Text** only (ablation) | `_text_embeddings.pkl` |
| C | QA Text + Audio, **Volatility Prompt** (proposed model) | `_qa_text_audio_prompt2_embeddings.pkl` |
| D | **QA Audio** only (ablation) | `_audio_embeddings.pkl` |
| E | **Transcript Chunks + Audio**, no QA units (ablation) | `_chunk_text_audio_embeddings.pkl` |

Each section writes one file per split (`kefvp_{train,dev,test}_{maec15,maec16}` and `kefvp_test_ec`). Section F attaches `ticker` and `date` to every record. Only section C is needed for Table I; Table III needs all five.

**Paper's runs:** Google Colab, NVIDIA A100 (about 15–20 hours for all five sections). Use an A100 or a GPU with as much memory to reproduce the released embeddings. On a GPU with less memory (for example an NVIDIA L4, 22 GB) a run can stop for lack of memory, and the embeddings can differ slightly from the released ones. The notebook is designed for Google Colab and was run on Colab only (README, "Where the notebooks run"). Embeddings computed again depend on the GPU and the library versions and may not be identical to the released ones; the released embeddings (data package, `embeddings/`) are the ones behind the paper's numbers.

**Inputs:** the QA files in `WORK_DIR/qa/` (the data package; for QA units from notebook 1, set `QA_DIR` to its output folder, `WORK_DIR/qa_generated`); the MAEC and EC audio (downloaded by this notebook: archives of 19.6 GB for MAEC and 4.7 GB for EC). **Outputs:** `WORK_DIR/embeddings/`, 35 files (5 variants × 7 splits); notebook 3 reads the 19 of them the paper uses.

### How to run

1. Edit the **settings cell** below if needed (run mode, folders). It is the only cell to edit.
2. Run all cells. To compute one variant only, run the cells up to section A, then that section's cells, then section F and the last cell.

Notebook 3 needs all 19 files in one folder, also when you computed one variant only: the other files then have to come from the data package. Section F saves every embedding file in the output folder again, also files of the data package; their content stays the same, but they no longer match the released checksums.

| `RUN_MODE` | What it does |
|---|---|
| `"smoke"` | Tests your setup: every cell, on the first 2 transcript chunks of each split. It still downloads the audio and the model. **The embeddings are a test output**, written to their own folder. Ends with `SMOKE TEST PASSED`. |
| `"full"` | The paper's run: every QA unit of every split, five variants. |

Run `"smoke"` first on a new setup. Each file is saved when its split is finished; a run that is interrupted loses the split in progress, and running a section again computes all its splits again.

Cell labels: **[SETUP]** folders, packages, downloads, run mode (no model or embedding code) · **[CHECK]** verifies files or results, changes nothing · **[EXPERIMENT]** the paper's code, unchanged; do not edit it when reproducing the paper.

## 0. Setup: data, paths, model

In [ ]:
# [SETUP] Settings: the only cell to edit. Nothing here changes the model, the prompts or the embeddings.

RUN_MODE = "full"
#   "full"  : the paper's run, every QA unit, five variants (about 15-20 hours on an A100)
#   "smoke" : tests your setup: every cell, on the first SMOKE_CHUNKS transcript chunks of each split

WORK_DIR = None
#   The folder for the inputs and the outputs. None = the default:
#     on Colab : /content/drive/MyDrive/qa-grounded-volatility   (Google Drive, as in the paper's runs)
#     elsewhere: qa-grounded-volatility-work, in the folder this notebook runs in
#   Any folder works. On Colab, a folder outside /content/drive is deleted when the session ends.

QA_DIR = None       # None = WORK_DIR/qa          (the data package; notebook 1 writes to WORK_DIR/qa_generated)
OUTPUT_DIR = None   # None = WORK_DIR/embeddings  ("smoke" writes to this name + "_smoke")
AUDIO_DIR = None    # where the audio archives (about 24 GB) are downloaded and unpacked (more again). None = the default:
#     on Colab : /content   (the runtime disk, as in the paper's runs; downloaded again in each session)
#     elsewhere: WORK_DIR/audio   (downloaded once, reused by later runs)

ARCHIVE_DIR = None  # optional: a folder that is kept (on Colab: on Google Drive) for the downloaded archives.
#   None: the archives are downloaded into AUDIO_DIR; on Colab that means again in every session, and Google Drive
#         can refuse a download for up to 24 hours when a file was fetched too often.
#   A folder, for example "/content/drive/MyDrive/qa-grounded-volatility/archives": the archives are downloaded
#         there once (about 24 GB: MAEC 19.6 GB, EC 4.7 GB) and every later session unpacks them from there
#         without downloading. A free Google Drive (15 GB) is too small for this.

SMOKE_CHUNKS = 2    # "smoke" mode: transcript chunks per split

# These settings can also be given as environment variables (for batch jobs); a variable that is set wins over
# this cell: QAV_RUN_MODE, QAV_WORK_DIR, QAV_QA_DIR, QAV_EMBEDDING_DIR (for OUTPUT_DIR; the name notebook 3 reads),
# QAV_AUDIO_DIR, QAV_ARCHIVE_DIR. SMOKE_CHUNKS: in this cell only.

In [ ]:
# [SETUP] Environment: resolves the folders and mounts Google Drive if a folder is on it.
import os
import shutil
import subprocess
import sys

RUN_MODE = os.environ.get("QAV_RUN_MODE") or RUN_MODE
assert RUN_MODE in ("full", "smoke"), f"RUN_MODE must be full or smoke, not {RUN_MODE!r}"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = os.environ.get("QAV_WORK_DIR", WORK_DIR) or (
    "/content/drive/MyDrive/qa-grounded-volatility" if IN_COLAB else "qa-grounded-volatility-work")
QA_DIR = os.environ.get("QAV_QA_DIR", QA_DIR) or os.path.join(WORK_DIR, "qa")
_output_root = os.environ.get("QAV_EMBEDDING_DIR", OUTPUT_DIR) or os.path.join(WORK_DIR, "embeddings")
# "smoke" never writes into the folder of a full run
EMBEDDING_DIR = _output_root if RUN_MODE == "full" else _output_root.rstrip("/\\") + "_smoke"
AUDIO_DIR = os.environ.get("QAV_AUDIO_DIR", AUDIO_DIR) or ("/content" if IN_COLAB else os.path.join(WORK_DIR, "audio"))

ARCHIVE_DIR = os.environ.get("QAV_ARCHIVE_DIR") or ARCHIVE_DIR or None
if ARCHIVE_DIR:
    ARCHIVE_DIR = os.path.abspath(ARCHIVE_DIR)

_folders = [os.path.abspath(p) for p in (WORK_DIR, QA_DIR, EMBEDDING_DIR, AUDIO_DIR)]
if IN_COLAB and any(p.startswith("/content/drive") for p in _folders + [ARCHIVE_DIR or ""]):
    from google.colab import drive
    drive.mount("/content/drive")
WORK_DIR, QA_DIR, EMBEDDING_DIR, AUDIO_DIR = _folders
os.makedirs(EMBEDDING_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
if ARCHIVE_DIR:
    os.makedirs(ARCHIVE_DIR, exist_ok=True)

print("run mode   :", RUN_MODE + (" (from the environment variable QAV_RUN_MODE)" if os.environ.get("QAV_RUN_MODE") else ""))
print("QA files   :", QA_DIR)
print("embeddings :", EMBEDDING_DIR)
print("audio      :", AUDIO_DIR)
print("archives   :", ARCHIVE_DIR or "in the audio folder (ARCHIVE_DIR is not set)")
print("environment:", "Google Colab" if IN_COLAB else "not Colab")
if RUN_MODE == "smoke":
    print(f"\n*** SMOKE TEST: the first {SMOKE_CHUNKS} transcript chunks of each split. This tests the setup. ***")
if IN_COLAB and not EMBEDDING_DIR.startswith("/content/drive") and RUN_MODE == "full":
    print("\nWARNING: the embeddings folder is on the Colab runtime and is deleted when the session ends.")
_have = [n for n in os.listdir(EMBEDDING_DIR) if n.startswith("kefvp_") and n.endswith(".pkl")]
if _have and RUN_MODE == "smoke":
    print(f"\nNOTE: the smoke folder holds {len(_have)} test files of an earlier smoke run. This run replaces them.")
elif _have:
    print(f"\nNOTE: the embeddings folder already holds {len(_have)} embedding files. Each section replaces the files "
          "of its variant, and section F saves every file again (same content, new checksum). To keep the files as "
          "they are, set OUTPUT_DIR to another folder.")

_QA_FILES = [f"kefvp_{s}_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
             for s in ("train_maec15", "dev_maec15", "test_maec15", "train_maec16", "dev_maec16", "test_maec16", "test_ec")
             ] + ["call_map.json"]
_missing = [n for n in _QA_FILES if not os.path.exists(os.path.join(QA_DIR, n))]
if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} of the {len(_QA_FILES)} QA files this notebook reads are not in {QA_DIR}:\n  " + "\n  ".join(_missing)
        + "\nPut the qa/ folder of the data package there (README, section 'Data'), "
          "or point QA_DIR in the settings cell to the folder that holds them (notebook 1 writes to qa_generated).")

In [ ]:
# [SETUP] Packages. transformers and bitsandbytes are pinned to the versions of the paper's runs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "python-dotenv", "gdown", "transformers==5.20.1", "bitsandbytes==0.50.2"], check=True)

import numpy as np
import requests
import time
from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv

try:
    import accelerate  # noqa: F401
    import librosa  # noqa: F401
    import PIL  # noqa: F401
    import torch
except ImportError as e:
    raise ImportError(f"{e}. Outside Colab, install the packages first: pip install -r requirements-embeddings.txt") from e
try:
    import torchvision  # noqa: F401     (the Qwen2.5-Omni processor does not load without it)
except ImportError as e:
    raise ImportError("torchvision is not installed, and the Qwen2.5-Omni processor needs it. Install the torchvision "
                      f"version that belongs to your PyTorch {torch.__version__} (https://pytorch.org/get-started/locally/); "
                      "do not let pip choose it alone, or it may replace your PyTorch.") from e
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU found. The model is loaded in 4-bit with bitsandbytes, which needs a CUDA GPU.")

In [ ]:
# [SETUP] Download and unpack the audio into AUDIO_DIR (the same sources and steps as in the paper's runs).
# Audio that is already unpacked in AUDIO_DIR is used as it is and not downloaded again. With ARCHIVE_DIR set,
# the downloaded archives are kept there and used again by later sessions, which then download nothing.
# Unpacking uses the unzip and zip commands on Linux (as on Colab) and 7-Zip on Windows.
import time

import gdown

if os.name == "nt" and len(AUDIO_DIR) > 120:
    print(f"WARNING: AUDIO_DIR is a long path ({len(AUDIO_DIR)} characters). Windows limits a file path to about 260 "
          "characters, and some audio files have long names: use a short folder such as D:/qav_audio.")
MAEC_dir = os.path.join(AUDIO_DIR, "MAEC_audio", "sp1500_earningscall")
original_data_dir = os.path.join(AUDIO_DIR, "original_dataset")
_archives = ARCHIVE_DIR or AUDIO_DIR          # where the downloaded archives are put


_WINDOWS = os.name == "nt"


def _seven_zip():
    """Windows: the 7-Zip program, which unpacks the archives there (it also reads the split EC archive)."""
    for _exe in (shutil.which("7z"), shutil.which("7z.exe"), r"C:\Program Files\7-Zip\7z.exe",
                 r"C:\Program Files (x86)\7-Zip\7z.exe"):
        if _exe and os.path.exists(_exe):
            return _exe
    raise RuntimeError("7-Zip is needed to unpack the archives on Windows and was not found. "
                       "Install it from https://www.7-zip.org and run this cell again.")


def _need_tools():
    if _WINDOWS:
        _seven_zip()
        return
    for _tool in ("zip", "unzip"):
        if shutil.which(_tool) is None:
            raise RuntimeError(f"the '{_tool}' command is needed to unpack the archives and was not found "
                               "(Linux: install the zip and unzip packages)")


def _run(cmd, cwd=None, quiet=False):
    subprocess.run(cmd, cwd=cwd, check=True, stdout=subprocess.DEVNULL if quiet else None)


def _unpack(zip_path, folder):
    """Unpacks into a temporary folder that gets its final name only when the unpack is complete."""
    _tmp = folder + ".unpacking"
    if os.path.exists(_tmp):
        shutil.rmtree(_tmp)   # left by an unpack that was interrupted
    if _WINDOWS:
        _run([_seven_zip(), "x", zip_path, "-o" + _tmp, "-y"], quiet=True)
    else:
        _run(["unzip", "-q", zip_path, "-d", _tmp])
    for _attempt in range(5):
        try:
            os.rename(_tmp, folder)
            break
        except PermissionError:           # Windows: another program (a virus scanner) can hold the new folder briefly
            if _attempt == 4:
                raise RuntimeError(f"the unpacked archive is in {_tmp}, but Windows did not allow renaming it to "
                                   f"{folder}. Close programs that use that folder, rename it by hand, and run this "
                                   "cell again.")
            time.sleep(5)


_DRIVE_HELP = ("Google Drive limits how often a file can be downloaded (the limit can last up to 24 hours); try again "
               "later. Or get the archive another way (in a browser; or open the file in Google Drive, choose 'Add "
               "shortcut to Drive' and copy it from your mounted Drive, see the README's Troubleshooting) and "
               "either unpack it into AUDIO_DIR under the folder name given above, or, for "
               f"the MAEC archive, put it in {_archives} as MAEC_audio.zip and run this cell again. To download "
               "the archives only once, set ARCHIVE_DIR in the settings cell to a folder that is kept.")

_folder = os.path.join(AUDIO_DIR, "MAEC_audio")
if os.path.exists(_folder):
    print("MAEC audio: found", _folder, "(not downloaded again)")
else:
    _need_tools()
    audio_file_id = "1m1GRCHgKn9Vz9IFMC_SpCog6uP3-gFgY"
    _zip = os.path.join(_archives, "MAEC_audio.zip")
    if os.path.exists(_zip):
        print("MAEC archive: found", _zip, "(not downloaded again)")
    else:
        try:
            gdown.download(f"https://drive.google.com/uc?id={audio_file_id}", output=_zip, quiet=False)
        except Exception as e:
            raise RuntimeError(f"the MAEC audio archive was not downloaded (target folder: {_folder}). " + _DRIVE_HELP) from e
        if not os.path.exists(_zip):
            raise RuntimeError(f"the MAEC audio archive was not downloaded (target folder: {_folder}). " + _DRIVE_HELP)
    try:
        _unpack(_zip, _folder)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"the MAEC archive {_zip} could not be unpacked; it may be incomplete or damaged. Delete "
                           "that file and run this cell again to download it again.") from e

# EC dataset (Qin & Yang 2019, ACL19 release): public Google Drive folder linked from
# github.com/GeminiLn/EarningsCall_Dataset (5 split zip parts, 4.7 GB).
if os.path.exists(original_data_dir):
    print("EC audio: found", original_data_dir, "(not downloaded again)")
else:
    _need_tools()
    folder_url = "https://drive.google.com/drive/folders/1BKCANORbcmUJKkOkBOghw6uNHPqS_az1"
    _zips = os.path.join(_archives, "original_zips")
    _complete = os.path.join(_zips, "download_complete")      # written when all parts are there
    if os.path.exists(_complete):
        print("EC archive: found", _zips, "(not downloaded again)")
    else:
        try:
            gdown.download_folder(url=folder_url, output=_zips, quiet=False)
        except Exception as e:
            raise RuntimeError(f"the EC archive was not downloaded (target folder: {original_data_dir}). " + _DRIVE_HELP) from e
        if _WINDOWS and any("ACL19_Release.zip" in files for _, _, files in os.walk(_zips)):
            open(_complete, "w").close()      # on Linux the marker is written below, once the parts are joined
    _part_dir = next((d for d, _, files in os.walk(_zips) if "ACL19_Release.zip" in files), None)
    if _part_dir is None:
        raise RuntimeError(f"the EC archive was not downloaded (target folder: {original_data_dir}). " + _DRIVE_HELP)
    try:
        if _WINDOWS:
            # 7-Zip reads the split parts (ACL19_Release.z01, ..., ACL19_Release.zip) directly
            _unpack(os.path.join(_part_dir, "ACL19_Release.zip"), original_data_dir)
        else:
            _zip = os.path.join(AUDIO_DIR, "ACL19_Release_All.zip")
            if os.path.exists(_zip):
                os.remove(_zip)
            _run(["zip", "-s", "0", "ACL19_Release.zip", "--out", _zip], cwd=_part_dir)   # joins the split parts
    except subprocess.CalledProcessError as e:
        if os.path.exists(_complete):
            os.remove(_complete)
        raise RuntimeError(f"the parts of the EC archive in {_zips} could not be joined or unpacked; a part may be "
                           "missing or incomplete. Run this cell again to download them again.") from e
    open(_complete, "w").close()
    if not _WINDOWS:
        try:
            _unpack(_zip, original_data_dir)
        except subprocess.CalledProcessError as e:
            os.remove(_complete)          # the parts are not trusted any more: the next run downloads them again
            raise RuntimeError(f"the EC archive joined from the parts in {_zips} could not be unpacked; a part may be "
                               "damaged. Run this cell again to download them again.") from e

for _expected in (MAEC_dir, os.path.join(original_data_dir, "ACL19_Release")):
    if not os.path.isdir(_expected):
        raise FileNotFoundError(f"{_expected} does not exist: the unpacked archive in AUDIO_DIR has another layout. "
                                "Delete its folder there and run this cell again.")
print("audio: ready in", AUDIO_DIR)
if ARCHIVE_DIR:
    print("archives kept in", ARCHIVE_DIR, "(MAEC_audio.zip, original_zips/): later sessions unpack them without downloading")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

INPUT_DIR = QA_DIR

kefvp_train_maec15_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_train_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

kefvp_dev_maec15_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_dev_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

kefvp_test_maec15_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_test_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

kefvp_train_maec16_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_train_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

kefvp_dev_maec16_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_dev_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

kefvp_test_maec16_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_test_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

print("All datasets loaded successfully.")
print(len(kefvp_train_maec15_chunks_qas),
      len(kefvp_dev_maec15_chunks_qas),
      len(kefvp_test_maec15_chunks_qas),
      len(kefvp_train_maec16_chunks_qas),
      len(kefvp_dev_maec16_chunks_qas),
      len(kefvp_test_maec16_chunks_qas))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

INPUT_DIR = QA_DIR

kefvp_test_ec_chunks_qas = json.load(open(os.path.join(
    INPUT_DIR,
    "kefvp_test_ec_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
), "r", encoding="utf-8"))

print(len(kefvp_test_ec_chunks_qas))

In [ ]:
# [SETUP] Audio folder and run mode. With AUDIO_DIR = /content (the Colab default) and RUN_MODE = "full",
# this cell changes nothing.
_SPLITS = ["kefvp_train_maec15", "kefvp_dev_maec15", "kefvp_test_maec15",
           "kefvp_train_maec16", "kefvp_dev_maec16", "kefvp_test_maec16", "kefvp_test_ec"]

# The QA files store the audio folders as they were in the paper's runs (/content/...). They are pointed to
# AUDIO_DIR here, in memory; the files on disk are not changed.
assert AUDIO_DIR not in ("/content/MAEC_audio", "/content/original_dataset"), "choose another AUDIO_DIR: this name is used inside it"
_n_moved = 0
for _split in _SPLITS:
    for _chunk in globals()[_split + "_chunks_qas"].values():
        _p = _chunk["audio_path"]
        for _sub in ("MAEC_audio", "original_dataset"):
            # a released path, and not one this cell already pointed to AUDIO_DIR (the cell can be run again)
            if _p.startswith(f"/content/{_sub}/") and not _p.startswith(os.path.join(AUDIO_DIR, _sub) + os.sep):
                _chunk["audio_path"] = os.path.join(AUDIO_DIR, *_p[len("/content/"):].split("/"))
                _n_moved += 1
if _n_moved:
    print(f"audio folders pointed to {AUDIO_DIR}: {_n_moved} transcript chunks")
_elsewhere = sorted({_chunk["audio_path"] for _split in _SPLITS for _chunk in globals()[_split + "_chunks_qas"].values()
                     if not os.path.normcase(os.path.abspath(_chunk["audio_path"])).startswith(
                         os.path.normcase(os.path.join(AUDIO_DIR, "")))})
if _elsewhere:
    print(f"NOTE: {len(_elsewhere)} audio folders named in the QA files are outside AUDIO_DIR, for example {_elsewhere[0]}.")
    print("      QA files written by notebook 1 name the audio folder of that run; the clips are read from there.")

if RUN_MODE == "smoke":
    for _split in _SPLITS:
        _name = _split + "_chunks_qas"
        globals()[_name] = dict(list(globals()[_name].items())[:SMOKE_CHUNKS])
    print(f"*** SMOKE TEST: {SMOKE_CHUNKS} transcript chunks per split ***")

# [CHECK] the audio clips the QA files refer to are in AUDIO_DIR
for _corpus, _folder in (("MAEC", "MAEC_audio"), ("EC", "original_dataset")):
    _n = _found = 0
    for _split in _SPLITS:
        if (_split == "kefvp_test_ec") != (_corpus == "EC"):
            continue
        for _chunk in globals()[_split + "_chunks_qas"].values():
            for _clip in _chunk["audio_map"].values():
                _n += 1
                _found += os.path.exists(os.path.join(_chunk["audio_path"], _clip))
    print(f"{_corpus} audio clips: {_found:,} of {_n:,} found")
    if _found == 0:
        raise FileNotFoundError(
            f"none of the {_corpus} audio clips was found. The QA files name these audio folders: under "
            f"{os.path.join(AUDIO_DIR, _folder)}" + (f", and outside AUDIO_DIR (for example {_elsewhere[0]}). For QA files "
            "written by notebook 1, set AUDIO_DIR to the AUDIO_DIR of that run." if _elsewhere else
            ". If that folder is incomplete or has another layout, delete it and run the download cell again."))
    if _found < _n:
        print("NOTE: a QA unit whose audio clip is missing is reported as [ERROR] by its section and gets no embedding.")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from transformers import (
    Qwen2_5OmniForConditionalGeneration,
    Qwen2_5OmniProcessor,
    BitsAndBytesConfig
)

import torch
import librosa

model_name = "Qwen/Qwen2.5-Omni-7B"
QWEN_REVISION = "ae9e1690543ffd5c0221dc27f79834d0294cba00"

# ==========================================
# Quantization config
# ==========================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    # best 4bit quant type
    bnb_4bit_quant_type="nf4",

    # compute in bf16 on GPU
    bnb_4bit_compute_dtype=torch.bfloat16,

    # extra compression
    bnb_4bit_use_double_quant=True,
)

# ==========================================
# Load quantized model
# ==========================================

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_name,
    revision=QWEN_REVISION,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True,

 )

processor = Qwen2_5OmniProcessor.from_pretrained(
    model_name,
    revision=QWEN_REVISION,
    trust_remote_code=True
)

model.eval()

Helpers: retry a failing batch, and save each embedding file atomically. When nothing fails, `safe_process_batch` is exactly `process_batch`.

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import os
import pickle

import torch

MAX_BATCH_ATTEMPTS = 3   # a failing batch is retried ...
MAX_UNIT_ATTEMPTS = 2    # ... then each of its units is embedded on its own
FAILED_UNITS = []        # units that could not be embedded (listed, then skipped)


def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_process_batch(process_batch, buffer):
    """process_batch(buffer); if it raises, retry the batch, then embed each unit alone. Results keep the buffer's
    order. A unit that still fails is listed in FAILED_UNITS and skipped."""
    last = None
    for attempt in range(MAX_BATCH_ATTEMPTS):
        try:
            results = process_batch(buffer)
            if len(results) != len(buffer):
                raise ValueError(f"{len(results)} results for {len(buffer)} units")
            return results
        except Exception as e:
            last = e
            print(f"[batch attempt {attempt + 1}/{MAX_BATCH_ATTEMPTS} failed] {type(e).__name__}: {e}")
            _free_gpu()
    results = []
    for unit in buffer:
        for attempt in range(MAX_UNIT_ATTEMPTS):
            try:
                results.extend(process_batch([unit]))
                break
            except Exception as e:
                last = e
                _free_gpu()
        else:
            unit_id = (unit["chunk_id"], unit.get("qa_idx"))
            FAILED_UNITS.append(unit_id)
            print(f"[unit skipped] {unit_id}: {type(last).__name__}: {last}")
    return results


def save_pickle(path, records):
    """Write the embedding list via a temporary file, so an interrupted save never leaves a broken file."""
    with open(path + ".tmp", "wb") as f:
        pickle.dump(records, f)
    os.replace(path + ".tmp", path)

## A. QA Text + Audio, General Prompt

### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
import torch
import librosa
import numpy as np
from tqdm.auto import tqdm
import gc
import os
import pickle

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import math
import gc
import pickle
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================
BATCH_SIZE = 4
SAVE_DIR = EMBEDDING_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_train_maec15": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16": kefvp_test_maec16_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================
def process_batch(batch_items):
    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:
        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]
        audio = item["audio"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an acoustic and emotional analysis assistant "
                    "specializing in Earnings Conference Calls (ECCs). "
                    "Focus primarily on how the speaker spoke "
                    "(tone, emotion, pace, confidence, hesitation, volatility), "
                    "rather than on the literal words."
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text":
                        f"Topic: {topic}\n\n"
                        f"Question: {q}\n\n"
                        f"Answer: {answer}\n\n"
                        f"Transcript:\n{raw_a}\n\n"
                        "Describe the emotional and tonal characteristics of this speech segment."
                    }
                ]
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "chunk_id": chunk_id,
            "qa_idx": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================
    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    last_hidden = outputs.hidden_states[-1]

    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_hidden = last_hidden * attention_mask

    embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

    embedding_np = embedding.detach().float().cpu().numpy()

    results = []
    for i in range(len(metadata)):
        results.append({
            "ecc_chunk_id": metadata[i]["chunk_id"],
            "qa_index": metadata[i]["qa_idx"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    del outputs, last_hidden, embedding, inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER DATASETS
# =========================================================
for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(chunk_items, desc=dataset_name):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):
            try:
                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""
                audio_segments = []

                for line in range(source_lines[0], source_lines[1] + 1):
                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(audio_file, sr=16000)
                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                    "audio": audio,
                })

                if len(batch_buffer) >= BATCH_SIZE:
                    batch_results = safe_process_batch(process_batch, batch_buffer)
                    all_embeddings.extend(batch_results)
                    batch_buffer = []

            except Exception as e:
                tqdm.write(f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================
    if len(batch_buffer) > 0:
        batch_results = safe_process_batch(process_batch, batch_buffer)
        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================
    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import math
import gc
import pickle
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================
BATCH_SIZE = 4
SAVE_DIR = EMBEDDING_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_test_ec": kefvp_test_ec_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================
def process_batch(batch_items):
    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:
        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]
        audio = item["audio"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an acoustic and emotional analysis assistant "
                    "specializing in Earnings Conference Calls (ECCs). "
                    "Focus primarily on how the speaker spoke "
                    "(tone, emotion, pace, confidence, hesitation, volatility), "
                    "rather than on the literal words."
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text":
                        f"Topic: {topic}\n\n"
                        f"Question: {q}\n\n"
                        f"Answer: {answer}\n\n"
                        f"Transcript:\n{raw_a}\n\n"
                        "Describe the emotional and tonal characteristics of this speech segment."
                    }
                ]
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "chunk_id": chunk_id,
            "qa_idx": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================
    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    last_hidden = outputs.hidden_states[-1]

    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_hidden = last_hidden * attention_mask

    embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

    embedding_np = embedding.detach().float().cpu().numpy()

    results = []
    for i in range(len(metadata)):
        results.append({
            "ecc_chunk_id": metadata[i]["chunk_id"],
            "qa_index": metadata[i]["qa_idx"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    del outputs, last_hidden, embedding, inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER DATASETS
# =========================================================
for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(chunk_items, desc=dataset_name):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):
            try:
                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""
                audio_segments = []

                for line in range(source_lines[0], source_lines[1] + 1):
                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(audio_file, sr=16000)
                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                    "audio": audio,
                })

                if len(batch_buffer) >= BATCH_SIZE:
                    batch_results = safe_process_batch(process_batch, batch_buffer)
                    all_embeddings.extend(batch_results)
                    batch_buffer = []

            except Exception as e:
                tqdm.write(f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================
    if len(batch_buffer) > 0:
        batch_results = safe_process_batch(process_batch, batch_buffer)
        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================
    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

## B. QA Text only

### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
import torch
import librosa
import numpy as np
from tqdm.auto import tqdm
import gc
import os
import pickle

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
### MAIN SCRIPT TO GENERATE AND SAVE EMBEDDINGS (BATCHED)

import math
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 4

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_train_maec15": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16": kefvp_test_maec16_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    metadata = []

    # =====================================================
    # BUILD INPUTS
    # =====================================================

    for item in batch_items:

        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an analysis assistant specializing in "
                    "Earnings Conference Calls (ECCs)."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Topic: {topic}\n\n"
                    f"Question: {q}\n\n"
                    f"Answer: {answer}\n\n"
                    f"Transcript:\n{raw_a}\n\n"
                    "Describe the characteristics of this answer."
                )
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)

        metadata.append({
            "ecc_chunk_id": chunk_id,
            "qa_index": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"]
        mask = attention_mask.unsqueeze(-1)

        masked_hidden = last_hidden * mask

        embedding = masked_hidden.sum(dim=1) / mask.sum(dim=1)

        embedding_np = embedding.detach().float().cpu().numpy()

    # =====================================================
    # UNPACK RESULTS
    # =====================================================

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "qa_index": metadata[i]["qa_index"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    # =====================================================
    # CLEANUP
    # =====================================================

    del outputs, last_hidden, embedding, inputs
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_text_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for local_chunk_idx, (chunk_id, chunk_data) in enumerate(
        tqdm(chunk_items, desc=dataset_name)
    ):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):

            try:

                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""

                # =====================================================
                # BUILD TEXT ONLY (NO AUDIO HERE)
                # =====================================================

                for line in range(source_lines[0], source_lines[1] + 1):
                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                # =====================================================
                # ADD TO BATCH BUFFER
                # =====================================================

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                })

                # =====================================================
                # PROCESS FULL BATCH
                # =====================================================

                if len(batch_buffer) >= BATCH_SIZE:

                    batch_results = safe_process_batch(process_batch, batch_buffer)

                    all_embeddings.extend(batch_results)

                    batch_buffer = []

            except Exception as e:

                tqdm.write(
                    f"\n[ERROR] chunk={chunk_id} qa={qa_idx} -> {e}"
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================

    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
### MAIN SCRIPT TO GENERATE AND SAVE EMBEDDINGS (BATCHED)

import math
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 4

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_test_ec": kefvp_test_ec_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    metadata = []

    # =====================================================
    # BUILD INPUTS
    # =====================================================

    for item in batch_items:

        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an analysis assistant specializing in "
                    "Earnings Conference Calls (ECCs)."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Topic: {topic}\n\n"
                    f"Question: {q}\n\n"
                    f"Answer: {answer}\n\n"
                    f"Transcript:\n{raw_a}\n\n"
                    "Describe the characteristics of this answer."
                )
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)

        metadata.append({
            "ecc_chunk_id": chunk_id,
            "qa_index": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"]
        mask = attention_mask.unsqueeze(-1)

        masked_hidden = last_hidden * mask

        embedding = masked_hidden.sum(dim=1) / mask.sum(dim=1)

        embedding_np = embedding.detach().float().cpu().numpy()

    # =====================================================
    # UNPACK RESULTS
    # =====================================================

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "qa_index": metadata[i]["qa_index"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    # =====================================================
    # CLEANUP
    # =====================================================

    del outputs, last_hidden, embedding, inputs
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_text_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for local_chunk_idx, (chunk_id, chunk_data) in enumerate(
        tqdm(chunk_items, desc=dataset_name)
    ):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):

            try:

                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""

                # =====================================================
                # BUILD TEXT ONLY (NO AUDIO HERE)
                # =====================================================
                for line in range(source_lines[0], source_lines[1] + 1):
                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                # =====================================================
                # ADD TO BATCH BUFFER
                # =====================================================

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                })

                # =====================================================
                # PROCESS FULL BATCH
                # =====================================================

                if len(batch_buffer) >= BATCH_SIZE:

                    batch_results = safe_process_batch(process_batch, batch_buffer)

                    all_embeddings.extend(batch_results)

                    batch_buffer = []

            except Exception as e:

                tqdm.write(
                    f"\n[ERROR] chunk={chunk_id} qa={qa_idx} -> {e}"
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================

    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

## C. QA Text + Audio, Volatility Prompt (proposed model)

### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 4

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_train_maec15": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16": kefvp_test_maec16_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:

        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]
        audio = item["audio"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a multimodal financial analysis assistant specializing in "
                    "earnings conference calls and stock volatility forecasting. "
                    "You are given a topic, a financial question, a transcript segment "
                    "and the corresponding audio. Analyze both what is said and how it is said. "
                    "Focus on signals relevant to future stock volatility."
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text":
                        f"Topic: {topic}\n\n"
                        f"Question: {q}\n\n"
                        f"Answer: {answer}\n\n"
                        f"ECC Segment:\n{raw_a}\n\n"
                        "Analyze this segment in context of volatility prediction."
                    }
                ]
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "ecc_chunk_id": chunk_id,
            "qa_index": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask

        embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

        embedding_np = embedding.detach().float().cpu().numpy()

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "qa_index": metadata[i]["qa_index"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    # cleanup
    del outputs, last_hidden, embedding, inputs
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(
        SAVE_DIR,
        f"{dataset_name}_qa_text_audio_prompt2_embeddings.pkl"
    )

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for local_chunk_idx, (chunk_id, chunk_data) in enumerate(
        tqdm(chunk_items, desc=dataset_name)
    ):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):

            try:

                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""
                audio_segments = []

                # =====================================================
                # BUILD TEXT + AUDIO
                # =====================================================

                for line in range(source_lines[0], source_lines[1] + 1):

                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(
                        audio_file,
                        sr=16000
                    )

                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                # =====================================================
                # ADD TO BATCH
                # =====================================================

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                    "audio": audio,
                })

                # =====================================================
                # PROCESS BATCH
                # =====================================================

                if len(batch_buffer) >= BATCH_SIZE:

                    batch_results = safe_process_batch(process_batch, batch_buffer)

                    all_embeddings.extend(batch_results)

                    batch_buffer = []

            except Exception as e:

                tqdm.write(
                    f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}"
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                continue

    # =========================================================
    # FINAL BATCH
    # =========================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(batch_results)

    # =========================================================
    # SAVE
    # =========================================================

    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 4

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_test_ec": kefvp_test_ec_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:

        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        answer = item["answer"]
        topic = item["topic"]
        raw_a = item["raw_a"]
        audio = item["audio"]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a multimodal financial analysis assistant specializing in "
                    "earnings conference calls and stock volatility forecasting. "
                    "You are given a topic, a financial question, a transcript segment "
                    "and the corresponding audio. Analyze both what is said and how it is said. "
                    "Focus on signals relevant to future stock volatility."
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text":
                        f"Topic: {topic}\n\n"
                        f"Question: {q}\n\n"
                        f"Answer: {answer}\n\n"
                        f"ECC Segment:\n{raw_a}\n\n"
                        "Analyze this segment in context of volatility prediction."
                    }
                ]
            }
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "ecc_chunk_id": chunk_id,
            "qa_index": qa_idx,
            "question": q,
            "topic": topic,
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask

        embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

        embedding_np = embedding.detach().float().cpu().numpy()

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "qa_index": metadata[i]["qa_index"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    # cleanup
    del outputs, last_hidden, embedding, inputs
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(
        SAVE_DIR,
        f"{dataset_name}_qa_text_audio_prompt2_embeddings.pkl"
    )

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for local_chunk_idx, (chunk_id, chunk_data) in enumerate(
        tqdm(chunk_items, desc=dataset_name)
    ):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):

            try:

                q = qa["Q"]
                a = qa["A"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""
                audio_segments = []

                # =====================================================
                # BUILD TEXT + AUDIO
                # =====================================================

                for line in range(source_lines[0], source_lines[1] + 1):

                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(
                        audio_file,
                        sr=16000
                    )

                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                # =====================================================
                # ADD TO BATCH
                # =====================================================

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "answer": a,
                    "topic": topic,
                    "raw_a": raw_a,
                    "audio": audio,
                })

                # =====================================================
                # PROCESS BATCH
                # =====================================================

                if len(batch_buffer) >= BATCH_SIZE:

                    batch_results = safe_process_batch(process_batch, batch_buffer)

                    all_embeddings.extend(batch_results)

                    batch_buffer = []

            except Exception as e:

                tqdm.write(
                    f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}"
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                continue

    # =========================================================
    # FINAL BATCH
    # =========================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(batch_results)

    # =========================================================
    # SAVE
    # =========================================================

    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

## D. QA Audio only

### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 4

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_train_maec15": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16": kefvp_test_maec16_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:
        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        topic = item["topic"]
        audio = item["audio"]

        # -------------------------------------------------
        # Audio ONLY (same prompt/model as the EC section)
        # -------------------------------------------------
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an acoustic analysis assistant. "
                    "Focus only on the acoustic properties of the speech."
                ),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "audio",
                        "audio": audio,
                    }
                ],
            },
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "chunk_id": chunk_id,
            "qa_idx": qa_idx,
            "question": q,
            "topic": topic,
        })

    # -------------------------------------------------
    # Processor
    # -------------------------------------------------
    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    last_hidden = outputs.hidden_states[-1]

    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_hidden = last_hidden * attention_mask

    embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

    embedding_np = embedding.detach().float().cpu().numpy()

    results = []
    for i in range(len(metadata)):
        results.append({
            "ecc_chunk_id": metadata[i]["chunk_id"],
            "qa_index": metadata[i]["qa_idx"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    del outputs, last_hidden, embedding, inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER DATASETS
# =========================================================
for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_audio_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(chunk_items, desc=dataset_name):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):
            try:
                q = qa["Q"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                audio_segments = []

                for line in range(source_lines[0], source_lines[1] + 1):
                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(audio_file, sr=16000)
                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "topic": topic,
                    "audio": audio,
                })

                if len(batch_buffer) >= BATCH_SIZE:
                    batch_results = safe_process_batch(process_batch, batch_buffer)
                    all_embeddings.extend(batch_results)
                    batch_buffer = []

            except Exception as e:
                tqdm.write(f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================
    if len(batch_buffer) > 0:
        batch_results = safe_process_batch(process_batch, batch_buffer)
        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================
    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import math
import gc
import pickle
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================
BATCH_SIZE = 4
SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_test_ec": kefvp_test_ec_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================
def process_batch(batch_items):
    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:
        chunk_id = item["chunk_id"]
        qa_idx = item["qa_idx"]
        q = item["question"]
        topic = item["topic"]
        audio = item["audio"]

        # -------------------------------------------------
        # Audio ONLY
        # -------------------------------------------------
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an acoustic analysis assistant. "
                    "Focus only on the acoustic properties of the speech."
                ),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "audio",
                        "audio": audio,
                    }
                ],
            },
        ]

        text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        texts.append(text)
        audios.append(audio)

        metadata.append({
            "chunk_id": chunk_id,
            "qa_idx": qa_idx,
            "question": q,
            "topic": topic,
        })

    # -------------------------------------------------
    # Processor
    # -------------------------------------------------
    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    last_hidden = outputs.hidden_states[-1]

    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_hidden = last_hidden * attention_mask

    embedding = masked_hidden.sum(dim=1) / attention_mask.sum(dim=1)

    embedding_np = embedding.detach().float().cpu().numpy()

    results = []
    for i in range(len(metadata)):
        results.append({
            "ecc_chunk_id": metadata[i]["chunk_id"],
            "qa_index": metadata[i]["qa_idx"],
            "question": metadata[i]["question"],
            "topic": metadata[i]["topic"],
            "embedding": embedding_np[i],
        })

    del outputs, last_hidden, embedding, inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER DATASETS
# =========================================================
for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(SAVE_DIR, f"{dataset_name}_audio_embeddings.pkl")

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(chunk_items, desc=dataset_name):

        qas = chunk_data["qas"]

        for qa_idx, qa in enumerate(qas):
            try:
                q = qa["Q"]
                topic = qa["topic"]
                source_lines = qa["source_lines"]

                raw_a = ""
                audio_segments = []

                for line in range(source_lines[0], source_lines[1] + 1):
                    raw_a += chunk_data["text_lines"][line - 1] + "\n"

                    audio_file = (
                        chunk_data["audio_path"] + "/" +
                        chunk_data["audio_map"][str(line)]
                    )

                    audio_array_chunk, _ = librosa.load(audio_file, sr=16000)
                    audio_segments.append(audio_array_chunk)

                audio = np.concatenate(audio_segments)

                batch_buffer.append({
                    "chunk_id": chunk_id,
                    "qa_idx": qa_idx,
                    "question": q,
                    "topic": topic,
                    "raw_a": raw_a,
                    "audio": audio,
                })

                if len(batch_buffer) >= BATCH_SIZE:
                    batch_results = safe_process_batch(process_batch, batch_buffer)
                    all_embeddings.extend(batch_results)
                    batch_buffer = []

            except Exception as e:
                tqdm.write(f"[ERROR] chunk={chunk_id}, qa={qa_idx} -> {e}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

    # =====================================================
    # FINAL BATCH
    # =====================================================
    if len(batch_buffer) > 0:
        batch_results = safe_process_batch(process_batch, batch_buffer)
        all_embeddings.extend(batch_results)

    # =====================================================
    # SAVE
    # =====================================================
    save_pickle(save_path, all_embeddings)

    print(f"Saved {len(all_embeddings)} embeddings to {save_path}")

## E. Transcript Chunks + Audio (no QA units)

Each chunk's full text and the concatenated audio of its lines; the audio processor keeps the first 300 s.

### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 2

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

datasets = {
    "kefvp_train_maec15": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16": kefvp_test_maec16_chunks_qas,
}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:

        texts.append(item["text"])
        audios.append(item["audio"])

        metadata.append({
            "ecc_chunk_id": item["chunk_id"]
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask

        embedding = (
            masked_hidden.sum(dim=1)
            / attention_mask.sum(dim=1)
        )

        embedding_np = embedding.detach().float().cpu().numpy()

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "embedding": embedding_np[i],
        })

    del outputs
    del last_hidden
    del embedding
    del inputs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(
        SAVE_DIR,
        f"{dataset_name}_chunk_text_audio_embeddings.pkl"
    )

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(
        chunk_items,
        desc=dataset_name
    ):

        try:

            ####################################################
            # BUILD FULL TRANSCRIPT
            ####################################################

            raw_text = "\n".join(chunk_data["text_lines"])

            ####################################################
            # BUILD FULL AUDIO
            ####################################################

            audio_segments = []

            for line in range(1, len(chunk_data["text_lines"]) + 1):

                audio_file = os.path.join(
                    chunk_data["audio_path"],
                    chunk_data["audio_map"][str(line)]
                )

                audio_array, _ = librosa.load(
                    audio_file,
                    sr=16000
                )

                audio_segments.append(audio_array)

            audio = np.concatenate(audio_segments)

            duration_seconds = len(audio) / 16000

            if duration_seconds > 300:
                tqdm.write(
                    f"[WARNING] chunk={chunk_id}: "
                    f"audio duration={duration_seconds:.1f}s "
                    f"({duration_seconds / 60:.1f} minutes)"
                )
            ####################################################
            # ADD TO BATCH — CHAT TEMPLATE
            ####################################################

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are an acoustic and emotional analysis assistant "
                        "specializing in Earnings Conference Calls (ECCs). "
                        "Focus primarily on how the speaker spoke "
                        "(tone, emotion, pace, confidence, hesitation, volatility), "
                        "rather than on the literal words."
                    )
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "audio", "audio": audio},
                        {
                            "type": "text",
                            "text": (
                                "Analyze the emotional and tonal characteristics "
                                "of this speech segment.\n\n"
                                f"Full transcript:\n{raw_text}"
                            )
                        }
                    ]
                }
            ]

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            batch_buffer.append({
                "chunk_id": chunk_id,
                "text": text,
                "audio": audio,
            })
            ####################################################
            # PROCESS BATCH
            ####################################################

            if len(batch_buffer) >= BATCH_SIZE:

                batch_results = safe_process_batch(process_batch, batch_buffer)

                all_embeddings.extend(
                    batch_results
                )

                batch_buffer = []

        except Exception as e:

            tqdm.write(
                f"[ERROR] chunk={chunk_id} -> {e}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            continue

    # =========================================================
    # FINAL BATCH
    # =========================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(
            batch_results
        )

    # =========================================================
    # SAVE
    # =========================================================

    save_pickle(save_path, all_embeddings)

    print(
        f"Saved {len(all_embeddings)} embeddings to {save_path}"
    )

### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import gc
import pickle
import numpy as np
import torch
import librosa
import os
from tqdm.auto import tqdm

# =========================================================
# CONFIG
# =========================================================

BATCH_SIZE = 2

SAVE_DIR = EMBEDDING_DIR

os.makedirs(SAVE_DIR, exist_ok=True)


datasets = {
    "kefvp_test_ec": kefvp_test_ec_chunks_qas,

}

# =========================================================
# HELPER
# =========================================================

def process_batch(batch_items):

    if len(batch_items) == 0:
        return []

    texts = []
    audios = []
    metadata = []

    for item in batch_items:

        texts.append(item["text"])
        audios.append(item["audio"])

        metadata.append({
            "ecc_chunk_id": item["chunk_id"]
        })

    # =====================================================
    # PROCESS BATCH
    # =====================================================

    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.thinker(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]

        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask

        embedding = (
            masked_hidden.sum(dim=1)
            / attention_mask.sum(dim=1)
        )

        embedding_np = embedding.detach().float().cpu().numpy()

    results = []

    for i in range(len(metadata)):

        results.append({
            "ecc_chunk_id": metadata[i]["ecc_chunk_id"],
            "embedding": embedding_np[i],
        })

    del outputs
    del last_hidden
    del embedding
    del inputs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# =========================================================
# MAIN LOOP OVER ALL DATASETS
# =========================================================

for dataset_name, dataset in datasets.items():

    print(f"\n========== Processing {dataset_name} ==========")

    save_path = os.path.join(
        SAVE_DIR,
        f"{dataset_name}_chunk_text_audio_embeddings.pkl"
    )

    all_embeddings = []
    batch_buffer = []

    chunk_items = list(dataset.items())

    for chunk_id, chunk_data in tqdm(
        chunk_items,
        desc=dataset_name
    ):

        try:

            ####################################################
            # BUILD FULL TRANSCRIPT
            ####################################################

            raw_text = "\n".join(chunk_data["text_lines"])

            ####################################################
            # BUILD FULL AUDIO
            ####################################################

            audio_segments = []

            for line in range(1, len(chunk_data["text_lines"]) + 1):

                audio_file = os.path.join(
                    chunk_data["audio_path"],
                    chunk_data["audio_map"][str(line)]
                )

                audio_array, _ = librosa.load(
                    audio_file,
                    sr=16000
                )

                audio_segments.append(audio_array)

            audio = np.concatenate(audio_segments)

            duration_seconds = len(audio) / 16000

            if duration_seconds > 300:
                tqdm.write(
                    f"[WARNING] chunk={chunk_id}: "
                    f"audio duration={duration_seconds:.1f}s "
                    f"({duration_seconds / 60:.1f} minutes)"
                )

            ####################################################
            # ADD TO BATCH — CHAT TEMPLATE
            ####################################################

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are an acoustic and emotional analysis assistant "
                        "specializing in Earnings Conference Calls (ECCs). "
                        "Focus primarily on how the speaker spoke "
                        "(tone, emotion, pace, confidence, hesitation, volatility), "
                        "rather than on the literal words."
                    )
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "audio", "audio": audio},
                        {
                            "type": "text",
                            "text": (
                                "Analyze the emotional and tonal characteristics "
                                "of this speech segment.\n\n"
                                f"Full transcript:\n{raw_text}"
                            )
                        }
                    ]
                }
            ]

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            batch_buffer.append({
                "chunk_id": chunk_id,
                "text": text,
                "audio": audio,
            })

            ####################################################
            # PROCESS BATCH
            ####################################################

            if len(batch_buffer) >= BATCH_SIZE:

                batch_results = safe_process_batch(process_batch, batch_buffer)

                all_embeddings.extend(
                    batch_results
                )

                batch_buffer = []

        except Exception as e:

            tqdm.write(
                f"[ERROR] chunk={chunk_id} -> {e}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            continue

    # =========================================================
    # FINAL BATCH
    # =========================================================

    if len(batch_buffer) > 0:

        batch_results = safe_process_batch(process_batch, batch_buffer)

        all_embeddings.extend(
            batch_results
        )

    # =========================================================
    # SAVE
    # =========================================================

    save_pickle(save_path, all_embeddings)

    print(
        f"Saved {len(all_embeddings)} embeddings to {save_path}"
    )

## F. Attach ticker and date to every embedding

Notebook 3 joins embeddings to the volatility labels on `(ticker, date)`. Run this after all sections.

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import glob
import json
import pickle

with open(os.path.join(QA_DIR, "call_map.json"), encoding="utf-8") as f:
    call_map = json.load(f)

for path in sorted(glob.glob(os.path.join(EMBEDDING_DIR, "kefvp_*.pkl"))):
    corpus = "ec" if os.path.basename(path).startswith("kefvp_test_ec_") else "maec"
    with open(path, "rb") as f:
        records = pickle.load(f)
    for rec in records:
        call = call_map[corpus][rec["ecc_chunk_id"].split("-")[0]]
        rec["ticker"], rec["date"], rec["call_id"] = call["ticker"], call["date"], call["text_file_name"]
    save_pickle(path, records)
    print(f"tagged {len(records):6d} records  {os.path.basename(path)}")

In [ ]:
# [CHECK] The files this run wrote. Changes nothing.
import pickle

_VARIANTS = {"A": "_embeddings.pkl", "B": "_text_embeddings.pkl", "C": "_qa_text_audio_prompt2_embeddings.pkl",
             "D": "_audio_embeddings.pkl", "E": "_chunk_text_audio_embeddings.pkl"}
_bad = []
for _section, _suffix in _VARIANTS.items():
    for _split in _SPLITS:
        _data = globals()[_split + "_chunks_qas"]
        _units = len(_data) if _section == "E" else sum(len(c["qas"]) for c in _data.values())
        _path = os.path.join(EMBEDDING_DIR, _split + _suffix)
        if not os.path.exists(_path):
            print(f"  {_section}  {_split + _suffix:62s} not written")
            _bad.append(_split + _suffix)
            continue
        with open(_path, "rb") as f:
            _records = pickle.load(f)
        _ok = len(_records) > 0 and all("ticker" in r and "date" in r and r["embedding"].shape == (3584,) for r in _records)
        print(f"  {_section}  {_split + _suffix:62s} {len(_records):6d} embeddings for {_units:6d} "
              f"{'chunks' if _section == 'E' else 'QA units'}{'' if _ok else '   <-- empty, or no ticker/date (run section F)'}")
        if not _ok:
            _bad.append(_split + _suffix)
print(f"units that could not be embedded after retries (FAILED_UNITS): {len(FAILED_UNITS)}")
print("A file can hold fewer embeddings than units: a few released QA units have an unusable evidence span (docs/DATA.md).")

if RUN_MODE == "smoke":
    if _bad:
        raise RuntimeError(f"the smoke run did not produce {len(_bad)} of the 35 files. Smoke mode expects all five sections "
                           f"and section F to have run (use Run all). Missing or incomplete: {_bad}")
    print("\nSMOKE TEST PASSED: every cell ran on this setup.")
    print(f"  35 test files written to {EMBEDDING_DIR} (they can be deleted)")
    print('  Next: set RUN_MODE = "full" in the settings cell and run all cells again.')
elif _bad:
    print(f"\n{len(_bad)} of the 35 files are missing or incomplete (expected if you ran only some sections).")
else:
    print(f"\nAll 35 files are in {EMBEDDING_DIR}. Notebook 3 reads 19 of them from there.")